# Notebook 5 — Results Assembly, Robustness & Replication Artifact
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

The final notebook. Pure assembly — no collection. It pulls every result produced by NB1–NB4 into **paper-ready tables and figures** organized around the honest three-part narrative, runs the remaining robustness checks, and packages a **replication artifact** (a single zip) for submission.

**The paper's three contributions (the narrative spine):**
1. **A validated method** for detecting AI-coding-tool adoption from public repository signals — G1: precision 1.0 (95% CI [0.975, 1.0]), recall 1.0 (CI [0.975, 1.0]), Cohen's κ = 0.991, on a 250-repo dual-coded gold set.
2. **A cautionary measurement result** on carbon attribution — a naive difference-in-differences attributes a large CI-compute increase (+135%) to adoption, but the parallel-trends assumption is violated (treated repos were already on a rising CI trajectory); once trends are accounted for, the effect is **null** (level jump +11%, p=0.65) or at most a non-significant suggestion (slope +48%, p=0.074). Activity outcomes (commits, PRs) show **no** effect. Naive AI-carbon attribution overstates the footprint several-fold.
3. **A bounded, descriptive carbon footprint** of the adopter cohort — **0.10 / 1.86 / 6.41 tCO₂e/yr** (low/central/high), CCF/SCI/Green-Algorithms-grounded, framed as association not causation.

**Standards:** ACM SIGSOFT *Repository Mining*, *Data Science*, *Longitudinal* — explicit sampling frame, validated constructs, frozen pre-registration + deviations log, full artifact release.


---
## Cell 0 — Re-attach environment & load all result artifacts


In [ ]:

%pip install -q pandas pyarrow numpy matplotlib

import json, pathlib, datetime, shutil, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT=pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS={k:PROJECT_ROOT/k for k in ['panels','cohort','results','figures','frame','gold']}
CONFIG=json.loads((PROJECT_ROOT/'config.json').read_text())

def load_json(p): 
    p=pathlib.Path(p); return json.loads(p.read_text()) if p.exists() else None
def load_csv(p):
    p=pathlib.Path(p); return pd.read_csv(p) if p.exists() else None

G1      = load_json(DIRS['results']/'G1_gate.json')
claim   = load_json(DIRS['results']/'reframed_claim.json')
balance = load_csv(DIRS['results']/'balance_table.csv')
did     = load_csv(DIRS['results']/'did_main.csv')
het     = load_csv(DIRS['results']/'heterogeneity_ci_minutes.csv')
carbon  = load_csv(DIRS['results']/'carbon_band_adopter_cohort.csv')
gap     = load_csv(DIRS['results']/'attribution_gap.csv')
print("Loaded artifacts:",
      {k:(v is not None) for k,v in
       dict(G1=G1,claim=claim,balance=balance,did=did,het=het,carbon=carbon,gap=gap).items()})


---
## Cell 1 — Table 1: G1 detector-validation (Contribution 1)

The measurement-validity result that licenses everything downstream. Reported with Wilson CIs (not bare perfection) and κ, per the agreed framing.


In [ ]:

t1 = pd.DataFrame([{
  "Metric":"Precision","Value":G1["precision"],"95% CI":f'[{G1["precision_95CI"][0]}, {G1["precision_95CI"][1]}]'},
  {"Metric":"Recall","Value":G1["recall"],"95% CI":f'[{G1["recall_95CI"][0]}, {G1["recall_95CI"][1]}]'},
  {"Metric":"F1","Value":G1["f1"],"95% CI":"-"},
  {"Metric":"Cohen's kappa","Value":G1["cohens_kappa"],"95% CI":"-"},
  {"Metric":"Confusion (TP/FP/FN/TN)",
   "Value":f'{G1["confusion"]["TP"]}/{G1["confusion"]["FP"]}/{G1["confusion"]["FN"]}/{G1["confusion"]["TN"]}',
   "95% CI":"-"},
])
t1.to_csv(DIRS['results']/'TABLE1_g1_validation.csv', index=False)
print("TABLE 1 — Adoption-detector validation (G1 gate):")
print(t1.to_string(index=False))
print("\nFraming for paper: report Wilson CI lower bounds (0.975), not 'perfect'; "
      "recall scoped as 'recall w.r.t. human-detectable adoption evidence'; "
      "construct reliably judgeable (kappa 0.991, n=242 decided pairs, 8 UNSURE adjudicated).")


---
## Cell 2 — Table 2: covariate balance + Table 3: DiD results (Contribution 2)

Balance shows residual imbalance on stars/pre-commits (disclosed). The DiD table shows the **naive** activity + CI effects; the trend-adjusted results (from the reframe) are appended to make the attribution caution explicit in one place.


In [ ]:

print("TABLE 2 — Covariate balance (SMD before/after matching):")
print(balance.to_string(index=False))
print("  Note: log_stars & log_pre_commits remain >0.1 after matching (residual imbalance "
      "from adopters being larger/more active) -> disclosed; motivates trend-adjusted analysis.")

# Table 3: naive DiD + trend-adjusted summary in one place
k=claim["key_numbers"]
t3 = pd.DataFrame([
  {"Outcome":"commits (activity)","Estimate %":k["commits_pct"],"p":k["commits_p"],"Spec":"naive DiD"},
  {"Outcome":"PRs (activity)","Estimate %":k["prs_pct"],"p":k["prs_p"],"Spec":"naive DiD"},
  {"Outcome":"CI-minutes","Estimate %":k["naive_did_ci_minutes_pct"],"p":"<0.01","Spec":"naive DiD (CONFOUNDED)"},
  {"Outcome":"CI-minutes","Estimate %":k["trend_adj_level_jump_pct"],"p":k["trend_adj_level_jump_p"],
   "Spec":"trend-adj LEVEL jump (NULL)"},
  {"Outcome":"CI-minutes","Estimate %":k["trend_adj_slope_pct"],"p":k["trend_adj_slope_p"],
   "Spec":"trend-adj SLOPE (n.s.)"},
  {"Outcome":"CI-minutes (placebo)","Estimate %":k["placebo_slope_pct"],"p":k["placebo_slope_p"],
   "Spec":"placebo (null = good)"},
  {"Outcome":"pre-trend itself","Estimate %":k["pre_trend_per_period_pct"],"p":k["pre_trend_p"],
   "Spec":"treated rising BEFORE adoption"},
])
t3.to_csv(DIRS['results']/'TABLE3_did_summary.csv', index=False)
print("\nTABLE 3 — DiD summary (naive vs trend-adjusted; the cautionary core):")
print(t3.to_string(index=False))


---
## Cell 3 — Table 4: carbon band + Table 5: attribution gap (Contribution 3)


In [ ]:

print("TABLE 4 — Descriptive carbon footprint of adopter cohort (association):")
print(carbon[['scenario','total_CI_minutes','energy_kWh','CO2e_tonnes_annualized']].to_string(index=False))
lo,ce,hi = carbon.set_index('scenario')['CO2e_tonnes_annualized'][['low','central','high']]
print(f"  -> BAND: {lo:.2f}-{hi:.2f} tCO2e/yr (central {ce:.2f}). Modest in absolute terms (anti-hype).")

print("\nTABLE 5 — Attribution gap (naive vs trend-adjusted, central scenario):")
print(gap[['estimate','effect_pct','attributable_tCO2e_central']].to_string(index=False))
naive=gap.iloc[0]['attributable_tCO2e_central']; adj=gap.iloc[1]['attributable_tCO2e_central']
print(f"  -> naive attributes {naive:.3f} t vs trend-adj {adj:.3f} t "
      f"(~{naive/adj:.0f}x overstatement). The paper's cautionary headline.")
carbon.to_csv(DIRS['results']/'TABLE4_carbon_band.csv', index=False)
gap.to_csv(DIRS['results']/'TABLE5_attribution_gap.csv', index=False)


---
## Cell 4 — Robustness summary

Consolidates the validity checks already run across notebooks into one reviewer-facing table: placebo (null), pre-trend test (violated → motivated the reframe), trend-adjustment outcome, retention-artifact correction, and the matching-balance caveat. Plus a sensitivity note on the carbon band width.


In [ ]:

robust = pd.DataFrame([
 {"Check":"Placebo (controls, fake adoption)","Result":"null (slope -5.2%, p=0.68)","Verdict":"PASS (no spurious effect)"},
 {"Check":"Parallel-trends (event-study leads)","Result":"violated (cleaned far-leads F=4.54, p=0.012)","Verdict":"FAIL -> reframe"},
 {"Check":"Trend-adjusted level jump","Result":"+11%, p=0.65","Verdict":"NULL"},
 {"Check":"Trend-adjusted slope (acceleration)","Result":"+48%, p=0.074","Verdict":"NOT SIG (suggestive)"},
 {"Check":"Retention-artifact correction","Result":"truncated months masked; pre-trend partly artifact","Verdict":"APPLIED"},
 {"Check":"Covariate balance (stars, pre-commits)","Result":"SMD ~0.30 after matching","Verdict":"DISCLOSED (motivates trend-adj)"},
 {"Check":"Carbon band width","Result":"0.10-6.41 tCO2e/yr (grid intensity dominant)","Verdict":"REPORTED as band, not point"},
 {"Check":"CI-minutes as compute proxy","Result":"wall-clock, not metered (CCF/SCI/GA)","Verdict":"DISCLOSED w/ known error bounds"},
])
robust.to_csv(DIRS['results']/'TABLE6_robustness.csv', index=False)
print("TABLE 6 — Robustness & validity summary:")
print(robust.to_string(index=False))


---
## Cell 5 — Headline figure (multi-panel) for the paper


In [ ]:

fig,ax=plt.subplots(1,3,figsize=(15,4.3))

# Panel A: event study (load if saved)
ev_path=DIRS['results']/'event_study_ci_minutes.csv'
if ev_path.exists():
    ev=pd.read_csv(ev_path)
    ax[0].axhline(0,color='gray',lw=.8); ax[0].axvline(-0.5,color='red',ls='--',lw=.8)
    ax[0].errorbar(ev['event_month'],ev['beta'],
                   yerr=[ev['beta']-ev['ci_low'],ev['ci_high']-ev['beta']],fmt='o-',capsize=3)
    ax[0].set_title('(A) CI-minutes event study\n(rising pre-trend = confound)')
    ax[0].set_xlabel('months rel. adoption'); ax[0].set_ylabel('DiD coef (log)')

# Panel B: naive vs trend-adjusted effect
k=claim['key_numbers']
labels=['naive\n+135%','adj level\n(null)','adj slope\n(+48%,n.s.)']
vals=[k['naive_did_ci_minutes_pct'],k['trend_adj_level_jump_pct'],k['trend_adj_slope_pct']]
ax[1].bar(range(3),vals,color=['#de2d26','#9ecae1','#fdae6b'])
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(labels,fontsize=8)
ax[1].set_ylabel('% change CI-minutes'); ax[1].set_title('(B) Effect collapses\nunder trend adjustment')
ax[1].axhline(0,color='gray',lw=.8)

# Panel C: carbon band
cb=carbon.set_index('scenario')['CO2e_tonnes_annualized']
ax[2].bar(['low','central','high'],[cb['low'],cb['central'],cb['high']],
          color=['#9ecae1','#3182bd','#08519c'])
ax[2].set_ylabel('tCO2e / year'); ax[2].set_title('(C) Adopter-cohort CI carbon\n(bounded, association)')
for i,s in enumerate(['low','central','high']):
    ax[2].text(i,cb[s],f'{cb[s]:.2f}',ha='center',va='bottom',fontsize=9)

plt.tight_layout(); plt.savefig(DIRS['figures']/'FIGURE1_headline.png',dpi=150,bbox_inches='tight')
plt.show(); print("Saved -> figures/FIGURE1_headline.png")


---
## Cell 6 — Package the replication artifact

Bundles the pre-registration, deviations log, config, frozen requirements, the gold set + codebook, the (de-identified) panels, all result tables/figures, and manifests into a single zip for submission. Raw caches and tokens are excluded.


In [ ]:

import zipfile
art_dir = PROJECT_ROOT/'artifact'; art_dir.mkdir(exist_ok=True)
include = [
  'PREREGISTRATION.txt','DEVIATIONS_LOG.txt','config.json','requirements_frozen.txt',
  'manifest_nb01.json','manifest_nb02.json','manifest_nb03A.json','manifest_nb03B.json',
  'manifest_nb04.json',
  'gold/CODEBOOK.txt','gold/gold_set.parquet',
  'cohort/cohort.parquet','cohort/adoption_dates.parquet','cohort/covariates_matched.parquet',
  'panels/panel_windowed.parquet','panels/panel_ci_cleaned.parquet',
  'results/G1_gate.json','results/reframed_claim.json','results/carbon_factors.json',
  'results/balance_table.csv','results/did_main.csv','results/heterogeneity_ci_minutes.csv',
  'results/carbon_band_adopter_cohort.csv','results/attribution_gap.csv',
  'results/TABLE1_g1_validation.csv','results/TABLE3_did_summary.csv',
  'results/TABLE4_carbon_band.csv','results/TABLE5_attribution_gap.csv',
  'results/TABLE6_robustness.csv',
  'figures/event_study_ci_minutes.png','figures/carbon_band_and_gap.png',
  'figures/FIGURE1_headline.png',
]
zip_path = art_dir/'replication_artifact.zip'
written=0; missing=[]
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in include:
        f=PROJECT_ROOT/rel
        if f.exists(): z.write(f, rel); written+=1
        else: missing.append(rel)
# README for the artifact
readme = f"""REPLICATION ARTIFACT — Second-Order Compute Footprint of AI-Coding-Tool Adoption in OSS
Generated: {datetime.datetime.utcnow().isoformat()}Z

CONTENTS
  PREREGISTRATION.txt   frozen analysis plan (measurement-validity gate + RQs)
  DEVIATIONS_LOG.txt    append-only log of all post-freeze changes incl. the reframe
  config.json           sampling frame + all parameters (single source of truth)
  gold/                 G1 gold set + codebook (adoption-detection validation)
  cohort/, panels/      analysis-ready repo x month panels (activity full-window; CI cleaned)
  results/, figures/    all paper tables + figures
  requirements_frozen.txt  exact package versions

HEADLINE FINDINGS
  1. Validated adoption detector: precision 1.0 (95% CI 0.975-1.0), recall 1.0
     (CI 0.975-1.0), Cohen's kappa 0.991 (n=250 gold set, dual-coded).
  2. Naive DiD attributes +135% CI-minutes to adoption, but parallel trends is
     VIOLATED; effect is null under trend adjustment (level +11%, p=0.65; slope
     +48%, p=0.074). No effect on commit/PR volume. -> naive carbon attribution
     overstates several-fold.
  3. Descriptive CI-carbon of adopter cohort: 0.10-6.41 tCO2e/yr (central 1.86),
     CCF/SCI/Green-Algorithms grounded, reported as a band; association not causation.

NOT INCLUDED: raw HTTP cache, GitHub token (collection re-runs need a user token).
"""
(art_dir/'README.txt').write_text(readme)
with zipfile.ZipFile(zip_path,'a',zipfile.ZIP_DEFLATED) as z:
    z.write(art_dir/'README.txt','README.txt')

print(f"Replication artifact: {zip_path}  ({written} files, {zip_path.stat().st_size//1024} KB)")
if missing: print("  (not found, skipped):", missing)


---
## Cell 7 — Final manifest & study summary


In [ ]:

summary = {
 "study":"Second-Order Compute Footprint of AI-Coding-Tool Adoption in OSS",
 "track":"ICSE SEIS",
 "contributions":claim["contributions"],
 "headline":claim["headline"],
 "frame":{"total_repos":1173,"treated":340,"controls":833},
 "G1":{"precision":G1["precision"],"recall":G1["recall"],"kappa":G1["cohens_kappa"]},
 "carbon_band_tCO2e_yr":{"low":0.10,"central":1.86,"high":6.41},
 "artifact":"artifact/replication_artifact.zip",
 "timestamp":datetime.datetime.utcnow().isoformat()+"Z",
}
(PROJECT_ROOT/'STUDY_SUMMARY.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print("\n=== PIPELINE COMPLETE (NB1-NB5). Replication artifact packaged. ===")
print("Next: write the paper around the three contributions; all tables/figures in results/ & figures/.")
